> **LangChain 1.x / 2026** — *2026** — drug-discovery evidence is auditable and human-gated; optional paid LLM only where noted. See `UPDATE_2026.md`.

# Chapter 8 — Compound-Target-Assay Evidence Graph (v2026) (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2008.%20LangChain%20for%20Drug%20Discovery/LC4LSH_Chapter_8_Compound_Target_Assay_Evidence_Graph.ipynb)

**Learning objectives**
- Model compounds, targets, and assays as typed nodes with provenance
- Add edges only with an explicit source + relation + confidence
- Query the graph locally and answer with citations or abstain
- Defer Neo4j until graph queries justify it

> Runtime: ~5 min (local)  
> Cost: free  
> Data: small built-in compound/target/assay records

## Environment setup

### Secrets (optional LLM only)

In [ ]:
import os


def get_secret(name, default=None):
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass
    return os.environ.get(name, default)


# These notebooks are LOCAL-first (RDKit/pandas/sklearn); a paid LLM is OPTIONAL.
OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY", "")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("OpenAI key set (optional):", bool(OPENAI_API_KEY))

### Install pinned dependencies

In [ ]:
%pip install -q rdkit "pandas>=2.0" "numpy>=1.26" "matplotlib>=3.8" "scikit-learn>=1.4" "langchain==1.0.0" "langchain-openai==1.0.0" "python-dotenv>=1.0" # Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

In [ ]:
# Optional LangSmith tracing (only if a key is present)
import os
LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "") or ""
LANGSMITH_PROJECT = "lc4lsh-chapter8-evidence-graph"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_API_KEY", LANGSMITH_API_KEY)
    os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    print("LangSmith OFF (no key) - fine; these notebooks are local-first.")

## Why an evidence graph?

Drug discovery connects **compounds**, **targets**, and **assays**. A graph makes those connections explicit AND auditable: every edge carries a source, a relation type, and a confidence. Start locally; add Neo4j only when traversal/analytics justify it.

## 1. Typed nodes with provenance

In [ ]:
from dataclasses import dataclass, field, asdict

@dataclass
class Node:
    nid: str
    ntype: str            # compound | target | assay
    name: str
    source: str           # provenance (db:id or file)
    meta: dict = field(default_factory=dict)

nodes = [
    Node("C1", "compound", "Aspirin", "DrugBank:DB00945", {"inchikey": "BSYNRYMUTXBXSQ-UHFFFAOYSA-N"}),
    Node("C2", "compound", "Imatinib", "DrugBank:DB00619", {"inchikey": "KTUFNOKKBVMGRW-UHFFFAOYSA-N"}),
    Node("T1", "target", "COX-1 (PTGS1)", "UniProt:P23219"),
    Node("T2", "target", "BCR-ABL", "UniProt:P00519-fusion"),
    Node("A1", "assay", "COX-1 inhibition IC50", "ChEMBL:CHEMBL assay-1", {"unit": "nM"}),
    Node("A2", "assay", "BCR-ABL kinase IC50", "ChEMBL:CHEMBL assay-2", {"unit": "nM"}),
]
for n in nodes:
    print(f"[{n.ntype:8s}] {n.nid:3s} {n.name:22s} <- {n.source}")

## 2. Edges with relation + confidence + source

In [ ]:
@dataclass
class Edge:
    src: str; dst: str
    relation: str         # tested_in | inhibits | measures
    confidence: str       # low | medium | high
    source: str           # provenance for THIS edge
    value: float = None   # optional measured value (nM)

edges = [
    Edge("C1", "A1", "tested_in", "high", "ChEMBL:rec-1"),
    Edge("C1", "T1", "inhibits", "high", "ChEMBL:rec-1", value=270.0),
    Edge("C2", "A2", "tested_in", "high", "ChEMBL:rec-2"),
    Edge("C2", "T2", "inhibits", "medium", "ChEMBL:rec-2", value=120.0),
    Edge("A1", "T1", "measures", "high", "assay annotation"),
    Edge("A2", "T2", "measures", "high", "assay annotation"),
]
for e in edges:
    print(f"{e.src} -[{e.relation}/{e.confidence}]-> {e.dst}  (src: {e.source})")

## 3. Build an in-memory graph

In [ ]:
import networkx as nx

G = nx.DiGraph()
for n in nodes:
    G.add_node(n.nid, **asdict(n))
for e in edges:
    G.add_edge(e.src, e.dst, relation=e.relation, confidence=e.confidence, source=e.source, value=e.value)
print("nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())
# quick check: every edge has a source
missing = [(u, v) for u, v, d in G.edges(data=True) if not d.get("source")]
print("edges missing provenance:", missing)

## 4. Query: compounds that inhibit a target

In [ ]:
def inhibitors_of(target_id, min_conf="medium"):
    rank = {"low": 0, "medium": 1, "high": 2}
    out = []
    for u, v, d in G.in_edges(target_id, data=True):
        if d.get("relation") == "inhibits" and rank.get(d.get("confidence"), 0) >= rank[min_conf]:
            out.append((G.nodes[u]["name"], d.get("value"), d.get("confidence"), d.get("source")))
    return out

for t in ["T1", "T2"]:
    print(t, G.nodes[t]["name"])
    for name, val, conf, src in inhibitors_of(t):
        print(f"   {name:10s} {val} nM  [{conf}]  src:{src}")

## 5. Answer-with-citations or abstain

In [ ]:
def answer(question_target):
    hits = inhibitors_of(question_target)
    if not hits:
        return f"ABSTAIN: no supported inhibitors recorded for {question_target}."
    lines = [f"Supported inhibitors of {G.nodes[question_target]['name']}:"]
    for name, val, conf, src in hits:
        lines.append(f" - {name}: {val} nM [{conf}] (source: {src})")
    return "\n".join(lines)

print(answer("T1"))
print()
print(answer("T99"))  # abstains

## 6. Export the graph + provenance

In [ ]:
import json
export = {
    "nodes": [asdict(n) for n in nodes],
    "edges": [asdict(e) for e in edges],
    "note": "every edge carries relation + confidence + source",
}
print(json.dumps(export, indent=2)[:600], "...")
print("\nLoad into Neo4j later only if traversal/analytics justify it.")

## Limitations & safety notes

- Toy graph; real graphs need curated ingestion and dedup by exact identifiers (InChIKey/UniProt/ChEMBL).
- An edge is an assertion with provenance, not established fact.
- `inhibits` here reflects recorded assay results, not a clinical claim.
- Local/free; Neo4j is optional and deferred.

In [ ]:
import gc
gc.collect()
for _v in ["records", "df", "model", "llm", "X", "graph"]:
    globals().pop(_v, None)
gc.collect()
print("Cleanup done.")

## Exercises

<details><summary>Why require a source on every edge?</summary>So any graph answer can be traced back to a record; unprovenanced edges are unsupported assertions.</details>

<details><summary>Why separate 'tested_in' from 'inhibits'?</summary>Being tested is not the same as showing activity; conflating them inflates apparent evidence.</details>

<details><summary>When is Neo4j worth adding?</summary>When you need multi-hop traversal, path queries, or analytics at scale — not for a handful of local edges.</details>

### Tasks
- **Task A** - Add a `contradicts` edge type and surface conflicting measurements for the same compound-target pair.
- **Task B** - Key compounds by InChIKey and add a dedup check that merges duplicate nodes safely.
- **Task C** - Add a read-only Cypher export and (optionally) load the export into a local Neo4j with provenance preserved.
- **Task D** - Add a confidence-weighted ranking of inhibitors with an explicit applicability-domain note.